# SQL - Duplicates

Copy exact or conflicting-key records from a disposable PostgreSQL practice table. Placement can be scattered, grouped across multiple periods, or mixed.

**Instructions:** Run `prepare_data_sql_update.ipynb` or the stable SQL preparation script to restore the clean `practice_N.weather_hourly` table. Adjust the schema and configuration. The notebook changes **only the practice table**, never canonical data. Requires PostgreSQL and `psycopg`.

In [ ]:
from random import Random
import re
import pandas as pd
import psycopg
from psycopg import sql

# Configuration
DATABASE_NAME = "analytical_workflow_fluency"
PRACTICE_SCHEMA = "practice_014"  # Match prepare_data.py
PRACTICE_TABLE = "weather_hourly"
TIME_COLUMN = "time"
KEY_COLUMN = "time"  # Unique identifier; not necessarily the time column in business data.
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive local calendar date.
SEED = 814205

# Duplicates
DUPLICATE_KIND = "mixed"  # "exact", "conflicting", "mixed", "random"
DUPLICATE_PLACEMENT = "mixed"  # "scattered", "grouped", "mixed", "random"
DUPLICATE_RATE = (0.01, 0.04)  # Share of eligible original rows copied once.
DUPLICATE_GROUPS = 3
DUPLICATE_GROUPED_SHARE = 0.70
DUPLICATE_CONFLICT_COLUMN = "temperature_2m"
DUPLICATE_CONFLICT_OFFSET = 1.5
DUPLICATE_CONFLICT_SHARE = 0.50

In [ ]:
# Shared SQL helpers: copy these only once in a combined dirty_data.py.
def sql_date_bounds(timezone, start_date, end_date):
    start = pd.Timestamp(start_date, tz=timezone).to_pydatetime()
    end = pd.Timestamp(end_date, tz=timezone).to_pydatetime()
    if start >= end:
        raise ValueError("Start date must precede end date.")
    return start, end


def check_practice_table(conn, schema):
    if not re.fullmatch(r"practice_[0-9]+", schema):
        raise ValueError("Only practice_N schemas can be deliberately corrupted.")
    if conn.autocommit:
        raise ValueError("Use a transaction (autocommit must be off).")

def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_duplicates_sql(
    conn, rng, *, schema, table, time_column, key_column, timezone,
    start_date, end_date, duplicate_kind, placement, duplicate_rate,
    n_groups, grouped_share, conflict_column, conflict_offset, conflict_share,
):
    check_practice_table(conn, schema)
    if duplicate_kind not in ("exact", "conflicting", "mixed", "random"):
        raise ValueError("Unrecognized duplicate kind.")
    if not 0 < conflict_share < 1:
        raise ValueError("Conflict share must be between 0 and 1.")
    if conflict_column in (time_column, key_column):
        raise ValueError("Conflict column cannot be the time or key column.")

    start, end = sql_date_bounds(timezone, start_date, end_date)
    source = sql.Identifier(schema, table)
    key_id = sql.Identifier(key_column)
    time_id = sql.Identifier(time_column)
    conflict_id = sql.Identifier(conflict_column)

    rows = conn.execute(
        sql.SQL("""
            SELECT {key}, {conflict}
            FROM {source}
            WHERE {time} >= %s AND {time} < %s
            ORDER BY {time}, {key}
        """).format(key=key_id, conflict=conflict_id, source=source, time=time_id),
        (start, end),
    ).fetchall()
    keys = [row[0] for row in rows]
    if not keys or any(k is None for k in keys) or len(set(keys)) != len(keys):
        raise ValueError("Expected non-NULL unique keys before duplication.")

    positions, chosen_placement = select_positions(
        len(keys), rng, method=placement, rate=duplicate_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    selected_keys = [keys[i] for i in positions]
    chosen_kind = rng.choice(("exact", "conflicting", "mixed")) if duplicate_kind == "random" else duplicate_kind
    if chosen_kind == "mixed" and len(selected_keys) < 2:
        raise ValueError("Mixed duplicate kinds need at least two copied records.")
    conflict_count = (len(selected_keys) if chosen_kind == "conflicting" else
                      max(1, min(len(selected_keys) - 1, round(len(selected_keys) * conflict_share)))
                      if chosen_kind == "mixed" else 0)
    conflict_keys = rng.sample(selected_keys, k=conflict_count)
    original = dict(rows)
    if conflict_keys and (conflict_offset == 0 or any(original[k] is None for k in conflict_keys)):
        raise ValueError("Conflicting copies need non-NULL numeric values and a nonzero offset.")

    # Only explicitly listed writable columns are copied. CTAS practice tables are ordinary columns.
    columns = conn.execute("""
        SELECT column_name, data_type, is_identity, is_generated
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position
    """, (schema, table)).fetchall()
    if not columns or any(identity != "NO" or generated != "NEVER" for _, _, identity, generated in columns):
        raise ValueError("Expected an ordinary writable practice table.")
    names = [name for name, *_ in columns]
    if any(name not in names for name in (key_column, time_column, conflict_column)):
        raise ValueError("Required columns are missing.")
    if conflict_count:
        type_name = dict((name, kind) for name, kind, *_ in columns)[conflict_column]
        numeric = {"smallint", "integer", "bigint", "numeric", "real", "double precision", "decimal"}
        if type_name not in numeric:
            raise ValueError("Conflict column must be numeric.")
        if type_name in {"smallint", "integer", "bigint"} and float(conflict_offset) % 1:
            raise ValueError("Use an integer offset for integer columns.")

    expressions = []
    for name in names:
        identifier = sql.Identifier(name)
        if name == conflict_column and conflict_count:
            expressions.append(sql.SQL(
                "CASE WHEN {key} = ANY(%s) THEN {value} + %s ELSE {value} END"
            ).format(key=key_id, value=identifier))
        else:
            expressions.append(identifier)

    query = sql.SQL("""
        INSERT INTO {source} ({columns})
        SELECT {expressions} FROM {source}
        WHERE {key} = ANY(%s) AND {time} >= %s AND {time} < %s
        RETURNING {key}, {conflict}
    """).format(
        source=source, columns=sql.SQL(", ").join(sql.Identifier(c) for c in names),
        expressions=sql.SQL(", ").join(expressions),
        key=key_id, time=time_id, conflict=conflict_id,
    )
    params = ((conflict_keys, conflict_offset) if conflict_count else ()) + (selected_keys, start, end)
    inserted = conn.execute(query, params).fetchall()
    if len(inserted) != len(selected_keys) or {k for k, _ in inserted} != set(selected_keys):
        raise ValueError("Inserted keys do not match selection; roll back transaction.")
    for key, value in inserted:
        if (key in conflict_keys) != (value != original[key]):
            raise ValueError("Unexpected copied-value behavior; roll back transaction.")
    return len(inserted), chosen_kind, chosen_placement


In [ ]:
# Small business-data distinction: repeated order keys may agree or conflict.
orders = pd.DataFrame({"order_id": [101, 102, 103], "amount": [30.0, 70.0, 90.0]})
copy = orders.iloc[[1, 2]].copy()
copy.iloc[1, copy.columns.get_loc("amount")] += 5
example = pd.concat([orders, copy], ignore_index=True)
assert example.duplicated(subset=["order_id"]).sum() == 2
assert example.duplicated().sum() == 1
example

# Seeded selection behavior, independently of the database.
for trial_method in ("scattered", "grouped", "mixed", "random"):
    a, _ = select_positions(240, Random(SEED), method=trial_method,
                            rate=(0.05, 0.05), n_groups=3, grouped_share=0.70)
    b, _ = select_positions(240, Random(SEED), method=trial_method,
                            rate=(0.05, 0.05), n_groups=3, grouped_share=0.70)
    assert a == b and len(a) == 12


In [ ]:
# Savepoint check: a failed combined corruption must roll back fully.
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    source = sql.Identifier(PRACTICE_SCHEMA, PRACTICE_TABLE)
    before = conn.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(source)).fetchone()[0]
    try:
        with conn.transaction():
            rng = Random(SEED)
            added, kind, placement = add_duplicates_sql(
                conn, rng, schema=PRACTICE_SCHEMA, table=PRACTICE_TABLE,
                time_column=TIME_COLUMN, key_column=KEY_COLUMN, timezone=TIMEZONE,
                start_date=START_DATE, end_date=END_DATE,
                duplicate_kind=DUPLICATE_KIND, placement=DUPLICATE_PLACEMENT,
                duplicate_rate=DUPLICATE_RATE, n_groups=DUPLICATE_GROUPS,
                grouped_share=DUPLICATE_GROUPED_SHARE, conflict_column=DUPLICATE_CONFLICT_COLUMN,
                conflict_offset=DUPLICATE_CONFLICT_OFFSET, conflict_share=DUPLICATE_CONFLICT_SHARE,
            )
            raise RuntimeError("Rollback test")
    except RuntimeError as exc:
        if str(exc) != "Rollback test":
            raise
    after = conn.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(source)).fetchone()[0]
    assert after == before
print("Rollback verified")

In [ ]:
# Apply within one database transaction; no commits occur inside the function.
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    rng = Random(SEED)
    added, kind, placement = add_duplicates_sql(
        conn, rng, schema=PRACTICE_SCHEMA, table=PRACTICE_TABLE,
        time_column=TIME_COLUMN, key_column=KEY_COLUMN, timezone=TIMEZONE,
        start_date=START_DATE, end_date=END_DATE,
        duplicate_kind=DUPLICATE_KIND, placement=DUPLICATE_PLACEMENT,
        duplicate_rate=DUPLICATE_RATE, n_groups=DUPLICATE_GROUPS,
        grouped_share=DUPLICATE_GROUPED_SHARE, conflict_column=DUPLICATE_CONFLICT_COLUMN,
        conflict_offset=DUPLICATE_CONFLICT_OFFSET, conflict_share=DUPLICATE_CONFLICT_SHARE,
    )
    print("Extra records:", added, "| kind:", kind, "| placement:", placement)

In [ ]:
# Inspect the actual repeated business keys (not only identical whole rows).
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    table_id = sql.Identifier(PRACTICE_SCHEMA, PRACTICE_TABLE)
    key_id = sql.Identifier(KEY_COLUMN)
    time_id = sql.Identifier(TIME_COLUMN)
    repeated = conn.execute(sql.SQL("""
        SELECT {key}, COUNT(*) AS copies FROM {table}
        GROUP BY {key} HAVING COUNT(*) > 1 ORDER BY copies DESC, {key} LIMIT 20
    """).format(key=key_id, table=table_id)).fetchall()
repeated

In [ ]:
# Compare output with the known clean pre-corruption state in development.
# A duplicate key may represent legitimate history in a different table grain.
assert added > 0
# No new key should occur outside the configured scope (requires a source-side count for a full audit).

In [ ]:
# Developer diagnostic: extra observations per key.
import matplotlib.pyplot as plt
if repeated:
    counts = pd.DataFrame(repeated, columns=[KEY_COLUMN, "copies"])
    ax = counts.set_index(KEY_COLUMN)["copies"].sub(1).plot.bar(figsize=(8, 3))
    ax.set_ylabel("Extra records")
    ax.set_title("Repeated business keys (sample)")
    plt.tight_layout()
    plt.show()